## 📦 Instalación y Setup

In [ ]:
# Instalar dependencias (ejecutar solo la primera vez)
# !pip install -r requirements.txt

In [1]:
# Imports principales
import os
import sys
from pathlib import Path
import json
import warnings
warnings.filterwarnings('ignore')

# Asegurar que el directorio raíz está en el path
ROOT_DIR = Path.cwd()
if str(ROOT_DIR) not in sys.path:
    sys.path.insert(0, str(ROOT_DIR))

print(f"📁 Working directory: {ROOT_DIR}")

📁 Working directory: c:\Users\c0d06h6\OneDrive - Walmart Inc\Cesar Delgado\--PERSONAL\Proyectos_Personales\13_Simpsons_Quote_Serch


In [ ]:
# Imports del proyecto
from config import get_settings, SYSTEM_PROMPTS
from utils import (
    normalize_text, clean_quote, tokenize_simple,
    reciprocal_rank_fusion, Timer,
    MAIN_CHARACTERS, normalize_character_name
)

settings = get_settings()
print(f"✅ Configuration loaded: {settings.APP_NAME} v{settings.APP_VERSION}")

---
## 1️⃣ Ingesta de Datos

Cargaremos quotes de Los Simpsons desde HuggingFace datasets.

In [ ]:
from ingestion import DataIngester, QuoteLoader
from ingestion.preprocessors import QuotePreprocessor

# Crear ingester
ingester = DataIngester(data_dir="data", output_dir="data/processed")

print("📊 Data Ingester initialized")

In [ ]:
# Cargar datos desde HuggingFace (esto puede tomar unos minutos)
# Limitamos a 10,000 muestras para el demo

with Timer("Data Ingestion") as timer:
    count = ingester.ingest_from_huggingface(
        dataset_name="jayantdocplix/simpsons-script-lines",
        max_samples=10000  # Ajustar según recursos disponibles
    )

print(f"\n✅ Loaded {count} quotes in {timer.elapsed_seconds:.2f}s")

In [ ]:
# Deduplicar y obtener estadísticas
duplicates_removed = ingester.deduplicate()
stats = ingester.get_statistics()

print(f"🔄 Removed {duplicates_removed} duplicates")
print(f"\n📈 Statistics:")
print(f"   - Total documents: {stats['total_documents']}")
print(f"   - Unique characters: {stats['unique_characters']}")
print(f"   - Unique seasons: {stats['unique_seasons']}")
print(f"\n🌟 Top 10 Characters:")
for char, count in stats['top_characters'][:10]:
    print(f"   - {char}: {count} quotes")

In [ ]:
# Guardar datos procesados
output_path = ingester.save_processed("quotes_processed.json")
print(f"💾 Data saved to: {output_path}")

In [ ]:
# Mostrar ejemplos de quotes
print("\n📝 Sample Quotes:")
print("=" * 60)

for i, doc in enumerate(ingester.documents[:5]):
    print(f"\n[{i+1}] {doc.character}:")
    print(f"    \"{doc.text}\"")
    if doc.episode_code:
        print(f"    📺 Episode: {doc.episode_code}")

---
## 2️⃣ Indexación y Búsqueda Híbrida

Construiremos índices BM25 y vectoriales para búsqueda híbrida.

In [ ]:
from retrieval import BM25Index, EmbeddingModel, HybridRetriever
from retrieval.embeddings import VectorStore

# Preparar documentos para indexación
documents = [doc.to_dict() for doc in ingester.documents]

print(f"📚 Preparing {len(documents)} documents for indexing...")

In [ ]:
# Crear retriever híbrido
retriever = HybridRetriever(
    bm25_weight=0.5,
    semantic_weight=0.5,
    rrf_k=60
)

retriever.initialize(lazy=False)  # Cargar modelo de embeddings ahora
print("✅ Hybrid Retriever initialized")

In [ ]:
# Indexar documentos (esto puede tomar varios minutos)
with Timer("Document Indexing") as timer:
    num_indexed = retriever.index_documents(
        documents,
        text_field="text",
        id_field="id",
        show_progress=True
    )

print(f"\n✅ Indexed {num_indexed} documents in {timer.elapsed_seconds:.2f}s")

In [ ]:
# Guardar índices para uso posterior
retriever.save("data/indices")
print("💾 Indices saved to data/indices/")

### 🔍 Probando la Búsqueda

In [ ]:
def display_search_results(results, title="Search Results"):
    """Muestra resultados de búsqueda de forma bonita."""
    print(f"\n🔍 {title}")
    print("=" * 60)
    
    for i, doc in enumerate(results[:5], 1):
        print(f"\n[{i}] Score: {doc.get('score', 0):.4f}")
        print(f"    👤 {doc.get('character', 'Unknown')}:")
        print(f"    \"{doc.get('text', '')[:100]}...\"" if len(doc.get('text', '')) > 100 else f"    \"{doc.get('text', '')}\"")
        if doc.get('episode_code'):
            print(f"    📺 {doc.get('episode_code')}")

In [ ]:
# Búsqueda BM25 (léxica)
query = "beer and donuts"

with Timer("BM25 Search") as timer:
    bm25_results = retriever.search(query, top_k=5, method="bm25")

display_search_results(bm25_results, f"BM25 Results for: '{query}'")
print(f"\n⏱️ Search time: {timer.elapsed_ms:.2f}ms")

In [ ]:
# Búsqueda Semántica
query = "Homer's philosophy about food and happiness"

with Timer("Semantic Search") as timer:
    semantic_results = retriever.search(query, top_k=5, method="semantic")

display_search_results(semantic_results, f"Semantic Results for: '{query}'")
print(f"\n⏱️ Search time: {timer.elapsed_ms:.2f}ms")

In [ ]:
# Búsqueda Híbrida (RRF)
query = "Bart's famous catchphrases and rebellious attitude"

with Timer("Hybrid Search") as timer:
    hybrid_results = retriever.search(query, top_k=5, method="hybrid")

display_search_results(hybrid_results, f"Hybrid Results for: '{query}'")
print(f"\n⏱️ Search time: {timer.elapsed_ms:.2f}ms")

---
## 3️⃣ RAG: Generación de Respuestas

Usaremos el contexto recuperado para generar respuestas con LLM.

In [ ]:
from retrieval.generator import ResponseGenerator

# Nota: Requiere OPENAI_API_KEY configurada
# Puedes configurarla así:
# os.environ['OPENAI_API_KEY'] = 'tu-api-key'

generator = ResponseGenerator(
    model="gpt-3.5-turbo",
    temperature=0.7,
    max_tokens=500
)

print("✅ Response Generator initialized")
print(f"   Model: {generator.model}")
print(f"   API Key configured: {'✅' if settings.OPENAI_API_KEY else '❌'}")

In [ ]:
def ask_springfield(question: str, top_k: int = 5):
    """
    Pregunta a Springfield - Pipeline RAG completo.
    """
    print(f"\n❓ Pregunta: {question}")
    print("=" * 60)
    
    # 1. Retrieval
    with Timer("Retrieval") as retrieval_timer:
        documents = retriever.search(question, top_k=top_k, method="hybrid")
    
    print(f"\n📚 Retrieved {len(documents)} documents in {retrieval_timer.elapsed_ms:.2f}ms")
    
    # 2. Generation
    with Timer("Generation") as gen_timer:
        response = generator.generate(question, documents)
    
    # 3. Mostrar respuesta
    print(f"\n💬 Respuesta:")
    print("-" * 40)
    print(response.get('answer', 'No response generated'))
    print("-" * 40)
    
    # 4. Métricas
    print(f"\n📊 Métricas:")
    print(f"   - Tokens usados: {response.get('tokens_used', 'N/A')}")
    print(f"   - Latencia total: {retrieval_timer.elapsed_ms + gen_timer.elapsed_ms:.2f}ms")
    print(f"   - Documentos usados: {len(documents)}")
    
    return response, documents

In [ ]:
# Ejemplo 1: Filosofía de Homero
response1, docs1 = ask_springfield(
    "¿Cuál es la filosofía de vida de Homero Simpson?"
)

In [ ]:
# Ejemplo 2: Catchphrases de Bart
response2, docs2 = ask_springfield(
    "What are Bart Simpson's most famous catchphrases?"
)

In [ ]:
# Ejemplo 3: Sabiduría de Lisa
response3, docs3 = ask_springfield(
    "¿Qué consejos de vida da Lisa Simpson?"
)

---
## 4️⃣ Evaluación

Evaluaremos la calidad del retrieval y las respuestas generadas.

In [ ]:
from eval import RetrievalEvaluator, ResponseEvaluator, ExperimentTracker
from eval.retrieval_metrics import RetrievalResult

print("✅ Evaluation modules loaded")

### 📊 Evaluación de Retrieval

In [ ]:
# Crear test set sintético
test_queries = [
    {
        "id": "q1",
        "query": "Homer eating donuts",
        "relevant_characters": ["Homer Simpson"]
    },
    {
        "id": "q2", 
        "query": "Bart skateboard trouble school",
        "relevant_characters": ["Bart Simpson"]
    },
    {
        "id": "q3",
        "query": "Lisa saxophone music",
        "relevant_characters": ["Lisa Simpson"]
    },
    {
        "id": "q4",
        "query": "Mr Burns evil money power",
        "relevant_characters": ["Mr. Burns"]
    },
    {
        "id": "q5",
        "query": "Ned Flanders neighborino church",
        "relevant_characters": ["Ned Flanders"]
    }
]

print(f"📋 Created {len(test_queries)} test queries")

In [ ]:
# Evaluar retrieval
evaluator = RetrievalEvaluator(k_values=[1, 3, 5, 10])

for test_case in test_queries:
    # Buscar
    results = retriever.search(test_case['query'], top_k=10, method="hybrid")
    
    # Determinar relevantes (basado en personaje)
    relevant_ids = [
        r['id'] for r in results 
        if r.get('character') in test_case['relevant_characters']
    ]
    
    # Añadir resultado
    evaluator.add_query_result(
        query_id=test_case['id'],
        query_text=test_case['query'],
        retrieved_ids=[r['id'] for r in results],
        retrieved_scores=[r['score'] for r in results],
        relevant_ids=relevant_ids if relevant_ids else [results[0]['id']]  # Fallback
    )

# Calcular métricas
metrics = evaluator.evaluate()

print(evaluator.summary())

### 📊 Evaluación de Respuestas

In [ ]:
# Evaluar respuestas generadas
response_evaluator = ResponseEvaluator(use_llm=False)  # Solo heurísticas

# Evaluar la primera respuesta
if response1 and docs1:
    score = response_evaluator.evaluate(
        question="¿Cuál es la filosofía de vida de Homero Simpson?",
        response=response1.get('answer', ''),
        documents=docs1
    )
    
    print("\n📊 Response Evaluation:")
    print(f"   - Faithfulness: {score.faithfulness:.3f}")
    print(f"   - Groundedness: {score.groundedness:.3f}")
    print(f"   - Relevance: {score.relevance:.3f}")
    print(f"   - Completeness: {score.completeness:.3f}")
    print(f"   - Overall: {score.overall:.3f}")

### 📈 Tracking de Experimentos

In [ ]:
from eval.experiments import SimpleTracker

# Usar tracker simple (sin MLflow)
tracker = SimpleTracker(output_dir="experiments")

# Registrar experimento
exp_id = tracker.start_experiment("hybrid_retrieval_eval")

tracker.log_params({
    "bm25_weight": 0.5,
    "semantic_weight": 0.5,
    "embedding_model": settings.EMBEDDING_MODEL,
    "num_documents": len(documents),
    "num_test_queries": len(test_queries)
})

tracker.log_metrics({
    "recall_at_5": metrics.recall_at_5,
    "mrr": metrics.mrr,
    "ndcg_at_10": metrics.ndcg_at_10
})

output_file = tracker.end_experiment()
print(f"\n💾 Experiment saved to: {output_file}")

---
## 5️⃣ Observabilidad

Demostración de logging estructurado y métricas.

In [ ]:
from observability import setup_structured_logging, get_logger
from observability.logging import log_search, log_generation, LogContext

# Configurar logging
setup_structured_logging(level="INFO", format_type="text")
logger = get_logger("notebook")

print("✅ Structured logging configured")

In [ ]:
# Ejemplo de logging estructurado
with LogContext(user_id="demo_user", session_id="notebook_123"):
    logger.info(
        "demo_search",
        query="test query",
        method="hybrid",
        num_results=5
    )

In [ ]:
from observability.metrics import PrometheusMetrics

# Inicializar métricas
PrometheusMetrics.initialize()

# Registrar algunas métricas de ejemplo
PrometheusMetrics.record_search(
    method="hybrid",
    num_results=5,
    latency_ms=150.5,
    success=True
)

PrometheusMetrics.set_index_size("bm25", len(documents))
PrometheusMetrics.set_index_size("vector", len(documents))

print("✅ Prometheus metrics recorded")

---
## 🎯 Demo Final: Pregúntale a Springfield

Interactúa con el sistema completo.

In [ ]:
# Demo interactivo
demo_questions = [
    "¿Qué frase resume mejor a Homero cuando le hablan de dieta?",
    "What does Homer think about work?",
    "¿Cuáles son las frases más memorables de Bart?",
    "What wisdom does Lisa share about life?",
    "¿Qué opina Mr. Burns sobre el dinero?"
]

print("🍩 Welcome to 'Ask Springfield'!")
print("=" * 60)
print("\nPrueba estas preguntas de ejemplo:")
for i, q in enumerate(demo_questions, 1):
    print(f"  {i}. {q}")

In [ ]:
# Ejecutar demo con la primera pregunta
response, docs = ask_springfield(demo_questions[0])

---
## 📝 Resumen y Próximos Pasos

### Lo que hemos construido:

1. ✅ **Ingesta de datos** desde HuggingFace datasets
2. ✅ **Indexación híbrida** con BM25 y embeddings semánticos
3. ✅ **Búsqueda híbrida** con Reciprocal Rank Fusion
4. ✅ **RAG** con citaciones usando OpenAI
5. ✅ **Evaluación** de retrieval (Recall, MRR, NDCG) y respuestas
6. ✅ **Observabilidad** con logging estructurado y métricas

### Próximos pasos sugeridos:

- 🐳 Desplegar con Docker (ver `docker-compose.yml`)
- 🌐 Probar la API REST en `http://localhost:8000/docs`
- 📊 Configurar MLflow para tracking avanzado
- 🔍 Experimentar con diferentes modelos de embeddings
- 🎨 Crear UI con Streamlit

---

**Autor:** César Adrián Delgado Díaz  
**Licencia:** MIT  

*"D'oh!" - Homer Simpson*